In [1]:
import pandas as pd
vesseles = pd.read_csv('../data/shadow_fleet_vessels.csv')

## 1. Capacity (DWT ranges)

In [2]:
size_DWT = {
    'Handysize':(7000,40000), #The lower limit has been lowered because we are taking annual ships into account
    'Aframax':(80000,110000),
    'Suezmax':(120000,200000),
}
vesseles['dwt_min'] = vesseles['tanker_size'].map(lambda s:size_DWT[s][0])
vesseles['dwt_max'] = vesseles['tanker_size'].map(lambda s:size_DWT[s][1])
vesseles['dwt_mid'] = (vesseles['dwt_min'] + vesseles['dwt_max']) / 2

vesseles[['vessel_name','tanker_size', 'dwt_min','dwt_max', 'dwt_mid']]


,vessel_name,tanker_size,dwt_min,dwt_max,dwt_mid
0,Aether,Aframax,80000,110000,95000.0
1,Tagor,Aframax,80000,110000,95000.0
2,Phoenix I,Aframax,80000,110000,95000.0
3,Bhilva,Aframax,80000,110000,95000.0
4,Torx,Suezmax,120000,200000,160000.0
5,Aura 1,Aframax,80000,110000,95000.0
6,Balu 6,Aframax,80000,110000,95000.0
7,Jewel,Aframax,80000,110000,95000.0
8,Guanyin,Aframax,80000,110000,95000.0
9,Marjorie,Suezmax,120000,200000,160000.0


In [3]:
#timeless values of voyages range
voyages_range = {
    'Handysize':(8,14),
    'Aframax':(5,8),
    'Suezmax':(4,7),
}
load_factor = 0.95
bbl_per_tone = 7.33

vesseles['voyages_min'] = vesseles['tanker_size'].map(lambda s:voyages_range[s][0])
vesseles['voyages_max'] = vesseles['tanker_size'].map(lambda s:voyages_range[s][1])

vesseles['tonnes_low'] = vesseles['dwt_min']*vesseles['voyages_min']*load_factor
vesseles['tonnes_high'] = vesseles['dwt_max']*vesseles['voyages_max']*load_factor

vesseles['bbl_low'] = vesseles['tonnes_low']*bbl_per_tone
vesseles['bbl_high'] = vesseles['tonnes_high']*bbl_per_tone

In [4]:
urals_price = 66.5

vesseles['value_low'] = vesseles['bbl_low']*urals_price
vesseles['value_high'] = vesseles['bbl_high']*urals_price

by_class = vesseles.groupby('tanker_size')[['value_low','value_high']].sum()

In [5]:
vesseles.head()

,imo,vessel_name,tanker_size,build_year,flag,ship_manager,registered_owner,ism_manager,list_source,dwt_min,dwt_max,dwt_mid,voyages_min,voyages_max,tonnes_low,tonnes_high,bbl_low,bbl_high,value_low,value_high
0,9328170,Aether,Aframax,2007,Unknown,UAE. Zulu Ships Management,UAE. Wrasse Ship Co,UAE. Zulu Ships Management,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
1,9282481,Tagor,Aframax,2005,Unknown,UAE. Zulu Ships Management,UAE. Blaire Charters Ltd,UAE. Tarabya Logistics Ltd,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
2,9236248,Phoenix I,Aframax,2002,Comoros,UAE. Triviality Shipping Llc,UAE. Celestial Overseas Co Ltd,UAE. Triviality Shipping Llc,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
3,9439383,Bhilva,Aframax,2010,Panama,Seychelles. Tika Shipping Ltd,Seychelles. Tika Shipping Ltd,China. Song Hua Jiang Shipmanagement,new_crude_2025,80000,110000,95000.0,5,8,380000.0,836000.0,2785400.0,6127880.0,185229100.0,407504020.0
4,9311610,Torx,Suezmax,2006,Panama,Seychelles. Shiva Shipping Ltd,Seychelles. Shiva Shipping Ltd,Singapore. Crest Maritime Pte Ltd,new_crude_2025,120000,200000,160000.0,4,7,456000.0,1330000.0,3342480.0,9748900.0,222274920.0,648301850.0


## 3. Port visits (GFW)
for ship Tagor


In [6]:
import requests


from utils import get_vessel_info, headers

raw_test = get_vessel_info(9282481)
for entry in raw_test['entries']:
    for rec in entry['selfReportedInfo']:
        print(rec['id'],rec['shipname'])

c0830f34f-ff65-a8f1-62c9-1f0e4b069ae5 BRITISH GANNET
e343b1da5-5b7a-de04-e72f-279e95210584 TAGOR
3b359ae29-977a-8747-4fe7-0f0f79269db8 TAGOR
97776be53-3f32-2f42-0439-c5cb44cc945d TAGOR
1ff0ec2ce-ef7a-75f9-7c67-38abeb4f9207 PIPER
a4ab02c0f-f6ed-739d-2413-54991555104c FRENCH NAVY WARSHIP
20c2c5af0-044e-1a07-536f-ebe21ab350b3 PRIAPUS
779363393-3c4a-27ae-e75c-568791c59948 TAGOR


In [7]:
url = "https://gateway.api.globalfishingwatch.org/v3/events"
params = {
    'datasets[0]':'public-global-port-visits-events:latest',
    'vessels[0]': 'e343b1da5-5b7a-de04-e72f-279e95210584',
    'start-date': '2025-01-01',
    'end-date': '2025-12-31',
    'limit': 100,
    'offset': 0
}
r = requests.get(url, headers= headers, params=params )
print (r.text)

{"metadata":{"datasets":["public-global-port-visits-events:v4.0"],"vessels":["e343b1da5-5b7a-de04-e72f-279e95210584"],"dateRange":{"from":"2025-01-01","to":"2025-12-31"}},"limit":100,"offset":0,"nextOffset":null,"total":15,"entries":[{"start":"2024-12-14T05:37:40.000Z","end":"2025-01-03T09:41:13.000Z","id":"80f50b20892a3dcdbebc6cdb37e84c89","type":"port_visit","position":{"lat":59.9233,"lon":28.5965},"regions":{"mpa":[],"eez":["5690"],"rfmo":["ICCAT","NAMMCO","ACAP","IWC","NASCO","ICES"],"fao":["27.3.d","27.3","27","27.3.d.32"],"majorFao":["27"],"eez12Nm":["5690"],"highSeas":[],"mpaNoTakePartial":[],"mpaNoTake":[]},"boundingBox":[28.59650657494041,59.92333973477929,28.600174237032302,59.91691358779298],"distances":{"startDistanceFromShoreKm":8,"endDistanceFromShoreKm":8,"startDistanceFromPortKm":0,"endDistanceFromPortKm":0},"vessel":{"id":"e343b1da5-5b7a-de04-e72f-279e95210584","name":"TAGOR","ssvid":"352002249","flag":"PAN","type":"other","nextPort":null},"port_visit":{"visitId":"3422

In [8]:
data = r.json()
first=data['entries'][0]

first.keys()

dict_keys(['start', 'end', 'id', 'type', 'position', 'regions', 'boundingBox', 'distances', 'vessel', 'port_visit'])

In [9]:
from pprint import pprint

pprint(first['port_visit'])

{'confidence': '4',
 'durationHrs': 484.05916666666667,
 'endAnchorage': {'anchorageId': '46969735',
                  'atDock': False,
                  'distanceFromShoreKm': '8',
                  'flag': 'RUS',
                  'id': 'rus-ustluga',
                  'lat': 59.91691358779298,
                  'lon': 28.600174237032302,
                  'name': None,
                  'topDestination': 'UST LUGA'},
 'intermediateAnchorage': {'anchorageId': '46969731',
                           'atDock': False,
                           'distanceFromShoreKm': '8',
                           'flag': 'RUS',
                           'id': 'rus-ustluga',
                           'lat': 59.92333973477929,
                           'lon': 28.59650657494041,
                           'name': None,
                           'topDestination': 'UST LUGA'},
 'startAnchorage': {'anchorageId': '46969731',
                    'atDock': False,
                    'distanceFromShoreKm': '

In [10]:
rows = []
for e in data ['entries']:
    anch = e['port_visit']['intermediateAnchorage']
    rows.append({
        'start': e['start'],
        'port': anch['topDestination'],
        'country': anch['flag'],
        'hours':e['port_visit']['durationHrs'],
        'at_dock': anch['atDock']
    })

visits = pd.DataFrame(rows)
visits

# Rough tanker port-stay heuristics (hours):
# <6    short stop (crew, supplies, pilot)
# 6-24  canal transit / bunkering
# 24-48 loading or discharge at berth
# 48-120 discharge + queue / STS transfer
# 120-360 waiting for orders / floating storage
# >360  repair / drydock / long idle

,start,port,country,hours,at_dock
0,2024-12-14T05:37:40.000Z,UST LUGA,RUS,484.059167,False
1,2025-01-04T21:50:35.000Z,PRIMORSK,RUS,28.352500,False
2,2025-01-06T05:14:57.000Z,PRIMORSK,RUS,36.007222,False
3,2025-01-11T23:48:06.000Z,SKAGEN,DNK,8.194167,False
4,2025-02-03T10:37:37.000Z,PORT SAID,EGY,17.010556,False
5,2025-02-15T16:15:21.000Z,MUNDRA,IND,114.372500,False
6,2025-02-25T09:30:25.000Z,FUJAIRAH,ARE,202.173333,False
7,2025-03-08T11:07:23.000Z,AL DUQUM OMAN,OMN,709.354167,False
8,2025-04-15T22:58:43.000Z,SUEZ,EGY,31.729444,False
9,2025-04-30T14:52:31.000Z,SKAGEN,DNK,15.313056,False


### Can `atDock` identify loadings?
No. `atDock` is `False` for all 15 port visits of Tagor, including Russian oil terminals
where the vessel must have loaded. The flag is not reliable in GFW data, so it is not used.

**How loadings are identified instead:**
- port country = `RUS`
- stay of roughly 24–48 hours (typical loading time for an Aframax)
- consecutive visits to the same port are counted as one loading (e.g. Primorsk, 4 and 6 Jan)

In [11]:
ru = visits[visits['country'] == 'RUS'].drop(columns=['at_dock'])
loadings = ru[(ru['hours'] >= 12) & (ru['hours'] <= 120)]
loadings = loadings[loadings['port']!=loadings['port'].shift()]
loadings

,start,port,country,hours
1,2025-01-04T21:50:35.000Z,PRIMORSK,RUS,28.352500
10,2025-05-15T09:55:00.000Z,UST LUGA,RUS,48.891111
14,2025-07-27T15:25:41.000Z,KOZMINO,RUS,40.325833


Tagor: **3 loadings** in 2025 (Primorsk ×2 visits = 1 loading).


In [12]:
len(loadings)

3

In [13]:
n_loadings = len(loadings)
tagor = vesseles[vesseles['vessel_name']=='Tagor'].iloc[0]

real_low = n_loadings*tagor['dwt_min']*load_factor*bbl_per_tone*urals_price
real_high =n_loadings*tagor['dwt_max']*load_factor*bbl_per_tone*urals_price

print(real_low/1e6, real_high/1e6)

111.13746 152.8140075


### Tagor: real loadings vs assumption
| | Loadings | Cargo value |
|---|---|---|
| Assumption (section 2) | 5–8 / year | $185–408M |
| GFW port visits | 3 in ~7 months (≈5 / year) | **$111–153M** |

The assumption was reasonable, but on the high side.

This is a **lower bound**:
- data ends on 27 Jul (AIS could be off after that)
- only 1 of Tagor's 8 GFW ids was used
- the 20-day Ust-Luga stay was not counted (possible STS)